In [22]:
# This notebook trains models using 10 repeated splits with wealth-exclusive features
import os
import gc
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor

LABEL = "EnergyPoor"

def train_one_model(train_df, model_key, model_path):
    """
    Train exactly one AutoGluon model (traditional or DL) and return predictor.
    model_key examples: "CAT", "GBM", "NN_TORCH"
    """
    hyperparameters = {model_key: {}}

    predictor = TabularPredictor(
        label=LABEL,
        problem_type="binary",
        eval_metric="accuracy",
        path=model_path,
        verbosity=0
    ).fit(
        train_data=train_df,
        presets="medium_quality",
        hyperparameters=hyperparameters,
        fit_weighted_ensemble=False,
        num_stack_levels=0
    )
    return predictor

In [23]:
def enforce_model_dtypes(df):
    df = df.copy()

    # -------- Category columns --------
    category_cols = [
        "hv270WealthIndex",
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
        "v717Occupation",
        "745aHouseOwnership",
    ]

    for col in category_cols:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # -------- Integer columns --------
    int8_cols = [
        "hv220AgeOfHead",
        "hv009FamilySize",
        "hv014NoOfChildren",
        "EnergyPoor",
        
    ]

    for col in int8_cols:
        if col in df.columns:
            df[col] = df[col].astype("int8")

    # -------- Float columns --------
    float_cols = [
        "HouseholdClusterElevation",
         "hv216HouseSize",
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype("float64")

    return df

In [24]:
import os
import pandas as pd
import gc

def train_country_with_saved_splits(country_name, splits_folder, model_folder,
                                   trad_model_key, dl_model_key="FASTAI",
                                   n_splits=10):
    
    trad_acc, dl_acc = [], []

    for i in range(1, n_splits + 1):
        # ---- Load saved split ----
        train_path = f"{splits_folder}/split{i}_train.parquet"

        train_df = pd.read_parquet(train_path)

        # ---- Enforce dtypes ----
        train_df = enforce_model_dtypes(train_df)
        
         # (wealth-exclusive) drop wealth index if needed
        WEALTH_COL = "hv270WealthIndex"
        if WEALTH_COL in train_df.columns:
            train_df = train_df.drop(columns=[WEALTH_COL])

        # ---- Safety check ----
        if LABEL not in train_df.columns:
            raise ValueError(
                f"{LABEL} not found in training dataframe for {country_name}, split {i}."
            )

        # ---- Model save paths ----
        trad_path = f"{model_folder}/split{i}/TRAD_{trad_model_key}/"
        dl_path   = f"{model_folder}/split{i}/DL_{dl_model_key}/"
        os.makedirs(trad_path, exist_ok=True)
        os.makedirs(dl_path, exist_ok=True)

        # ---- Train Traditional ----
        pred_tr = train_one_model(train_df, trad_model_key, trad_path)
        
        # ---- Train DL ----
        pred_dl = train_one_model(train_df, dl_model_key, dl_path)
       
        print(f"{country_name} split {i} training completed.")

        # ---- Cleanup ----
        del pred_tr
        del pred_dl
        del train_df
        gc.collect()

    return trad_acc, dl_acc

In [ ]:
SPLITS_ROOT = r"INDIA/DATA/PAIRTEST10"

country_trad = {
    "INDIA": "GBM",
}

for country, trad_key in country_trad.items():

    MODELS_ROOT = rf"{SPLITS_ROOT}\MODELS\WEALTH_EXCLUSIVE"

    # training only (evaluate code inside function is commented out)
    train_country_with_saved_splits(
        country_name=country,
        splits_folder=SPLITS_ROOT,
        model_folder=MODELS_ROOT,
        trad_model_key=trad_key,
        dl_model_key="FASTAI",
        n_splits=10
    )

    print(f"\nTraining completed for {country} (10 splits). Models saved in: {MODELS_ROOT}\n")